# P3 — Los caminos son mensajes

Esto conecta P1 (over-squashing) y P2 (conteo de caminos). Tras `g` capas, un mensaje de `i` llega a `j`
**una vez por cada camino de longitud g**. Así que `n_g(i,j) = (A^g)[i,j]` es exactamente cuántas copias de la
información de `i` llegan a `j` — todas forzadas en un vector fijo.

## 1. Cada camino entrega un mensaje

<img src="../figs-theory/es/E3a_path_is_message.svg" width="760"/>

## 2. Los mensajes se apilan en un vector

Cuando la pila supera la capacidad del vector, el receptor no puede separarlos — over-squashing.

<img src="../figs-theory/es/E3b_messages_stack.svg" width="760"/>

In [ ]:
# Cuenta mensajes crudos vs el conteo de-duplicado (cociente) hacia el objetivo.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
for d in [1,2,3]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'profundidad {d}: crudo {int(raw[d][:,t].sum()):4d}   de-duplicado (kQ/I) {int(eff[d][:,t].sum()):3d}')

## 3. El cociente kQ/I: colapsar caminos equivalentes

Imponer una relación fusiona caminos paralelos en una clase, bajando la multiplicidad.

<img src="../figs-theory/es/T5_quotient.svg" width="760"/>

In [ ]:
# kQ/I colapsa los dos caminos de longitud 2 del diamante de 2 a 1 (vía aiq-quivers).
from oversquash.ideal_bridge import build_quotient_plan
import numpy as np
ei = np.array([[0,0,1,2],[1,2,3,3]])   # 0->1, 0->2, 1->3, 2->3
plan = build_quotient_plan(ei, num_nodes=4, max_length=2)
print('multiplicidad cruda (0->3, longitud 2):', plan.raw_mult.get((0,3,2)))
print('efectiva (kQ/I)                       :', plan.effective_mult.get((0,3,2)))

## 4. El giro: cuándo fusionar ayuda y cuándo daña

Colapsar caminos paralelos solo ayuda cuando son verdaderamente redundantes. En nuestra tarea de recuperación
la multiplicidad **es señal**, así que colapsarla quita información — un resultado negativo que confirmamos en
los experimentos.

<img src="../figs-theory/es/E3c_signal_vs_noise.svg" width="760"/>

**Siguiente (P4):** conservar todos los caminos, pero *aprender* cuánto pesar cada uno. Eso es atención.